# L = l-1 + attn(l-1) + MLP(attn(l-1) + l-1)  

In [1]:
"""
SD3 Hook 동작 검증 스크립트
============================

목적:
  실제 실험(sd3_causal_path.py) 전에,
  hook이 올바른 위치에서 올바른 tensor를 잡고 있는지 1개 프롬프트로 확인.

검증 방식:
  L = L_minus_1 + delta_attn + delta_mlp  는 수학적 항등식.
  f32로 변환 후 계산하면 오차가 거의 0이어야 함.

블록 내부 흐름:

  [block 입력] hidden_states  (batch, 4096, 1536), float16
       │
       ├─── ① L_minus_1 캡처  ← block 시작 시점
       │
       ▼
  norm1 + attention 계산
       │
       ▼
  hidden_states = hidden_states + gate * attn_output
  (= L_minus_1 + delta_attn  =  L_after_attn)
       │
       ├─── ② L_after_attn 캡처  ← norm2.forward_pre_hook
       │         norm2의 입력 = attention residual이 이미 더해진 상태
       │         norm2 실행 직전(pre_hook)에 args_in[0]으로 잡음
       │
       ▼
  hidden_states = hidden_states + gate * ff(norm2(hidden_states))
  (= L_after_attn + delta_mlp  =  L_output)
       │
       └─── ③ L_output 캡처  ← block_output[1]  (image stream)

  분해식:
    delta_attn = L_after_attn - L_minus_1
    delta_mlp  = L_output     - L_after_attn
    항등식:  L_minus_1 + delta_attn + delta_mlp == L_output
"""

'\nSD3 Hook 동작 검증 스크립트\n============================\n\n목적:\n  실제 실험(sd3_causal_path.py) 전에,\n  hook이 올바른 위치에서 올바른 tensor를 잡고 있는지 1개 프롬프트로 확인.\n\n검증 방식:\n  L = L_minus_1 + delta_attn + delta_mlp  는 수학적 항등식.\n  f32로 변환 후 계산하면 오차가 거의 0이어야 함.\n\n블록 내부 흐름:\n\n  [block 입력] hidden_states  (batch, 4096, 1536), float16\n       │\n       ├─── ① L_minus_1 캡처  ← block 시작 시점\n       │\n       ▼\n  norm1 + attention 계산\n       │\n       ▼\n  hidden_states = hidden_states + gate * attn_output\n  (= L_minus_1 + delta_attn  =  L_after_attn)\n       │\n       ├─── ② L_after_attn 캡처  ← norm2.forward_pre_hook\n       │         norm2의 입력 = attention residual이 이미 더해진 상태\n       │         norm2 실행 직전(pre_hook)에 args_in[0]으로 잡음\n       │\n       ▼\n  hidden_states = hidden_states + gate * ff(norm2(hidden_states))\n  (= L_after_attn + delta_mlp  =  L_output)\n       │\n       └─── ③ L_output 캡처  ← block_output[1]  (image stream)\n\n  분해식:\n    delta_attn = L_after_attn - L_minus_1\n    delta_mlp  = L_output  

## Load Pipeline

In [2]:
import torch
from diffusers import StableDiffusion3Pipeline

DEVICE = "cuda"
DTYPE = torch.float16
MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"

pipe = StableDiffusion3Pipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)

/home/haeun/miniconda3/envs/C3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 9/9 [00:01<00:00,  4.78it/s]


In [3]:
# print(pipe)
# print("=" * 100)
print(pipe.transformer)

SD3Transformer2DModel(
  (pos_embed): PatchEmbed(
    (proj): Conv2d(16, 1536, kernel_size=(2, 2), stride=(2, 2))
  )
  (time_text_embed): CombinedTimestepTextProjEmbeddings(
    (time_proj): Timesteps()
    (timestep_embedder): TimestepEmbedding(
      (linear_1): Linear(in_features=256, out_features=1536, bias=True)
      (act): SiLU()
      (linear_2): Linear(in_features=1536, out_features=1536, bias=True)
    )
    (text_embedder): PixArtAlphaTextProjection(
      (linear_1): Linear(in_features=2048, out_features=1536, bias=True)
      (act_1): SiLU()
      (linear_2): Linear(in_features=1536, out_features=1536, bias=True)
    )
  )
  (context_embedder): Linear(in_features=4096, out_features=1536, bias=True)
  (transformer_blocks): ModuleList(
    (0-22): 23 x JointTransformerBlock(
      (norm1): AdaLayerNormZero(
        (silu): SiLU()
        (linear): Linear(in_features=1536, out_features=9216, bias=True)
        (norm): LayerNorm((1536,), eps=1e-06, elementwise_affine=False)
 

In [4]:
block_idx = 0
block = pipe.transformer.transformer_blocks[block_idx]

print(block.attn)

Attention(
  (to_q): Linear(in_features=1536, out_features=1536, bias=True)
  (to_k): Linear(in_features=1536, out_features=1536, bias=True)
  (to_v): Linear(in_features=1536, out_features=1536, bias=True)
  (add_k_proj): Linear(in_features=1536, out_features=1536, bias=True)
  (add_v_proj): Linear(in_features=1536, out_features=1536, bias=True)
  (add_q_proj): Linear(in_features=1536, out_features=1536, bias=True)
  (to_out): ModuleList(
    (0): Linear(in_features=1536, out_features=1536, bias=True)
    (1): Dropout(p=0.0, inplace=False)
  )
  (to_add_out): Linear(in_features=1536, out_features=1536, bias=True)
)


In [5]:
import torch
from diffusers import StableDiffusion3Pipeline
from functools import wraps
from collections import defaultdict

DEVICE   = "cuda"
DTYPE    = torch.float16
MODEL_ID = "stabilityai/stable-diffusion-3-medium-diffusers"

# ── 모델 로드 (이미 로드된 경우 재사용) ──────────────────────
if "pipe" not in dir():
    print("Loading pipeline...")
    pipe = StableDiffusion3Pipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

# ── 결과 저장 버퍼 ────────────────────────────────────────────
# block_idx → { L_minus_1, L_after_attn, L_output, delta_attn, delta_mlp }
captured = defaultdict(dict)

# hook 활성화 플래그 (리스트로 감싸서 클로저 안에서 수정 가능)
hook_active = [False]

# ──────────────────────────────────────────────────────────────
# 블록 패치
# ──────────────────────────────────────────────────────────────
already_patched_count = 0

for block_idx, block in enumerate(pipe.transformer.transformer_blocks):

    # Jupyter 재실행 시 중복 패치 방지
    # → 중복 패치되면 _tracked → _tracked → ... 무한 재귀(RecursionError) 발생
    if getattr(block.forward, "_is_patched", False):
        already_patched_count += 1
        continue

    original_block_forward = block.forward

    def _make_patched_forward(original_forward, idx, blk):
        @wraps(original_forward)
        def patched_forward(hidden_states, encoder_hidden_states, temb, *args, **kwargs):

            # hook 비활성 상태면 원본 그대로 실행
            if not hook_active[0]:
                return original_forward(hidden_states, encoder_hidden_states, temb, *args, **kwargs)

            # ────────────────────────────────────────────
            # ① L_minus_1: block에 들어온 image token stream
            #   shape: (batch, 4096, 1536)
            #     batch=2: CFG로 uncond + cond 두 배치가 함께 들어옴
            #     4096: 64×64 image patch tokens
            #     1536: hidden dim
            #   dtype: float16 (모델 연산 dtype 그대로)
            #   detach(): 그래디언트 추적 끊기
            #   clone(): 이후 in-place 연산에 의해 값이 바뀌지 않도록 복사
            # ────────────────────────────────────────────
            L_minus_1 = hidden_states.detach().clone()  # (batch, 4096, 1536), float16

            # ────────────────────────────────────────────
            # ② L_after_attn: attention residual 적용 후, MLP 이전 상태
            #
            #   왜 norm2 pre_hook으로 잡나?
            #   블록 내부 순서:
            #     hidden_states += gate * attn_output   ← attention residual 적용
            #     hidden_states += gate * ff(norm2(hidden_states))  ← MLP
            #
            #   norm2는 MLP 직전에 hidden_states를 입력으로 받음.
            #   즉 norm2의 입력값 = L_after_attn (attention은 끝, MLP는 아직)
            #
            #   register_forward_pre_hook:
            #     norm2.forward() 실행 직전에 콜백 호출
            #     args_in: norm2.forward에 전달되는 positional args의 tuple
            #     args_in[0]: 첫 번째 인자 = hidden_states = L_after_attn
            # ────────────────────────────────────────────
            norm2_input_buffer = {}   # L_after_attn을 임시 보관하는 버퍼

            def _capture_norm2_input(module, args_in):
                # 한 번만 캡처 (혹시 pre_hook이 여러 번 호출되는 경우 방지)
                if "L_after_attn" not in norm2_input_buffer and args_in:
                    # args_in[0] = norm2에 전달되는 hidden_states = L_after_attn
                    norm2_input_buffer["L_after_attn"] = args_in[0].detach().clone()

            norm2_hook_handle = blk.norm2.register_forward_pre_hook(_capture_norm2_input)

            try:
                # 원본 block forward 실행
                # (실행 중 norm2가 호출될 때 _capture_norm2_input 콜백이 발동)
                block_output = original_forward(
                    hidden_states, encoder_hidden_states, temb, *args, **kwargs
                )
            finally:
                # 에러 여부와 무관하게 반드시 hook 제거
                # (제거 안 하면 이후 모든 forward에서 계속 호출됨)
                norm2_hook_handle.remove()

            # ────────────────────────────────────────────
            # ③ L_output: block 최종 출력의 image token 부분
            #
            #   block 반환값 구조:
            #     일반 블록 (block 0~22, context_pre_only=False):
            #       tuple → (encoder_hidden_states, hidden_states)
            #                 └── text (333 tokens)  └── image (4096 tokens)
            #       → image stream = block_output[1]
            #
            #     마지막 블록 (block 23, context_pre_only=True):
            #       tensor → hidden_states만 반환 (text MLP 생략)
            #       → image stream = block_output 자체
            # ────────────────────────────────────────────
            if isinstance(block_output, tuple):
                L_output = block_output[1].detach()   # image stream
            else:
                L_output = block_output.detach()      # context_pre_only 블록

            # ────────────────────────────────────────────
            # delta 계산 및 저장
            #
            # ★ 반드시 .float() 먼저, 뺄셈은 나중에
            #
            #   잘못된 순서 (f16 뺄셈 → f32 변환):
            #     delta = (L_after_attn_f16 - L_minus_1_f16)  ← f16 rounding 오차 발생
            #     delta_f32 = delta.float()                    ← 이미 오차가 고정된 값을 f32로
            #     → recon ≠ L_output  (rel_sum ~ 1e-4 수준 오차)
            #
            #   올바른 순서 (f32 변환 → 뺄셈):
            #     L_minus_1_f32 = L_minus_1.float()    ← 먼저 f32
            #     L_after_attn_f32 = ...float()        ← 먼저 f32
            #     delta_attn = L_after_attn_f32 - L_minus_1_f32  ← f32끼리 뺄셈
            #     → recon = L_minus_1 + delta_attn + delta_mlp = L_output  (항등식 성립)
            #     → 오차 ≈ 0 (f32 machine epsilon ~1e-7 수준)
            # ────────────────────────────────────────────
            if "L_after_attn" in norm2_input_buffer:

                # f32 변환 먼저
                L_minus_1_f32    = L_minus_1.cpu().float()
                L_after_attn_f32 = norm2_input_buffer["L_after_attn"].cpu().float()
                L_output_f32     = L_output.cpu().float()

                # f32끼리 뺄셈 → 오차 없음
                delta_attn = L_after_attn_f32 - L_minus_1_f32    # attention 기여분
                delta_mlp  = L_output_f32     - L_after_attn_f32  # MLP 기여분

                captured[idx] = {
                    "L_minus_1":    L_minus_1_f32,     # block 입력
                    "L_after_attn": L_after_attn_f32,  # attention 적용 후 (= norm2 입력)
                    "L_output":     L_output_f32,      # block 출력
                    "delta_attn":   delta_attn,        # Δattn
                    "delta_mlp":    delta_mlp,         # Δmlp
                }

            return block_output

        patched_forward._is_patched = True   # 재실행 시 중복 패치 방지용 마킹
        return patched_forward

    block.forward = _make_patched_forward(original_block_forward, block_idx, block)

if already_patched_count > 0:
    print(f"⚠ {already_patched_count}개 블록이 이미 패치됨 — 스킵")
    print("  새 패치 코드를 적용하려면 커널을 재시작하세요")
else:
    print(f"✓ {len(pipe.transformer.transformer_blocks)}개 블록 패치 완료")

# ── Denoising step tracker ────────────────────────────────────
# transformer.forward 전체를 감싸서 step 번호 추적
# step 0일 때만 hook 활성화
if not getattr(pipe.transformer.forward, "_is_step_tracked", False):
    original_transformer_forward = pipe.transformer.forward
    denoising_step_counter = [0]

    @wraps(original_transformer_forward)
    def step_tracked_forward(*args, **kwargs):
        hook_active[0] = (denoising_step_counter[0] == 0)  # step 0만 캡처
        result = original_transformer_forward(*args, **kwargs)
        denoising_step_counter[0] += 1
        return result

    step_tracked_forward._is_step_tracked = True
    pipe.transformer.forward = step_tracked_forward
    print("✓ Step tracker 부착 완료")
else:
    print("⚠ Step tracker 이미 부착됨 — 스킵")
    denoising_step_counter[0] = 0

# ── 1개 프롬프트 실행 ─────────────────────────────────────────
denoising_step_counter[0] = 0   # 항상 리셋
print("\n1개 프롬프트 실행 중 (step 0만 캡처)...")
with torch.no_grad():
    _ = pipe(
        "a cat",
        num_inference_steps=28,
        generator=torch.Generator(DEVICE).manual_seed(42),
    )

# ──────────────────────────────────────────────────────────────
# 검증 1: Recon 오차
# L_minus_1 + delta_attn + delta_mlp == L_output 인가?
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*75}")
print("검증 1: 재구성 오차  (L_minus_1 + delta_attn + delta_mlp == L_output?)")
print(f"{'='*75}")
print(f"{'Blk':>4}  {'shape':>20}  {'err_sum':>12}  {'rel_sum':>10}  {'da/dm':>7}  판정")
print(f"{'-'*75}")

n_blocks = len(pipe.transformer.transformer_blocks)
pass_cnt = 0

for b in range(n_blocks):
    ctx_pre = getattr(pipe.transformer.transformer_blocks[b], "context_pre_only", False)
    if b not in captured:
        print(f"  {b:>3}  [캡처 안 됨]")
        continue

    c = captured[b]

    # 항등식 검증: L_minus_1 + delta_attn + delta_mlp
    recon   = c["L_minus_1"] + c["delta_attn"] + c["delta_mlp"]
    err_sum = (recon - c["L_output"]).abs().sum().item()
    rel_sum = err_sum / (c["L_output"].abs().sum().item() + 1e-8)

    shape       = tuple(c["L_minus_1"].shape)
    da_dm_ratio = c["delta_attn"].norm().item() / (c["delta_mlp"].norm().item() + 1e-8)

    if rel_sum < 1e-5:
        verdict = "✓ 완벽"
        pass_cnt += 1
    elif rel_sum < 1e-3:
        verdict = "△ 작은 오차"
    else:
        verdict = "✗ hook 위치 의심"

    ctx_str = " [ctx_pre_only]" if ctx_pre else ""
    print(f"  {b:>3}  {str(shape):>20}  "
          f"err={err_sum:.3e}  rel={rel_sum:.2e}  "
          f"da/dm={da_dm_ratio:.3f}  {verdict}{ctx_str}")

print(f"{'='*75}")
print(f"\n{pass_cnt}/{n_blocks} 블록 통과  (rel_sum < 1e-5 기준)")
print("""
오차 해석:
  f32 변환 후 뺄셈 → 항등식 오차 이론상 0.
  rel_sum < 1e-5  →  ✓  정상 (f32 machine epsilon 수준)
  rel_sum < 1e-3  →  △  작은 오차 (재검토 권장)
  rel_sum > 1e-3  →  ✗  hook 위치가 잘못됐을 가능성
""")

# ──────────────────────────────────────────────────────────────
# 검증 2: Scale range (sanity check)
# 목적: 데이터가 정상 범위인가 확인 (분석 핵심은 방향, 크기 아님)
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*75}")
print("검증 2: Scale range (sanity check)")
print(f"{'='*75}")
print(f"{'Blk':>4}  {'dtype':>7}  "
      f"{'L-1 max':>9}  {'L-1 mean':>9}  {'L-1 med':>9}  "
      f"{'da  max':>9}  {'da  mean':>9}  {'da  med':>9}  "
      f"{'dm  max':>9}  {'dm  mean':>9}  {'dm  med':>9}")
print(f"{'-'*110}")

for b in range(n_blocks):
    if b not in captured:
        continue
    c = captured[b]

    # dtype 흐름:
    #   hook 캡처 시점 → float16  (.detach().clone())
    #   저장 시점      → float32  (.float() 후 뺄셈)
    hook_dtype_str = str(DTYPE).replace("torch.", "")

    L_minus_1  = c["L_minus_1"]
    delta_attn = c["delta_attn"]
    delta_mlp  = c["delta_mlp"]

    # max:  outlier 포착, skewed 분포에서 과대 추정 가능
    # mean: 전체 평균, outlier에 다소 민감
    # med:  분포 중심, outlier 영향 없음 (skewed 분포에서 가장 신뢰)
    print(f"  {b:>3}  {hook_dtype_str:>7}  "
          f"{L_minus_1.abs().max().item():>9.3f}  "
          f"{L_minus_1.abs().mean().item():>9.4f}  "
          f"{L_minus_1.abs().median().item():>9.4f}  "
          f"{delta_attn.abs().max().item():>9.4f}  "
          f"{delta_attn.abs().mean().item():>9.5f}  "
          f"{delta_attn.abs().median().item():>9.5f}  "
          f"{delta_mlp.abs().max().item():>9.4f}  "
          f"{delta_mlp.abs().mean().item():>9.5f}  "
          f"{delta_mlp.abs().median().item():>9.5f}")

print(f"{'='*110}")
print("""
scale 해석 (sanity check 용도):
  이 테이블 목적: 데이터가 정상 범위인지 확인.
  실험 핵심은 크기(norm)가 아닌 방향(cosine similarity).

  max >> mean >> med  →  skewed (소수 token에 에너지 집중)
  max ≈  mean ≈  med  →  균등한 분포

  L-1이 깊은 블록일수록 커짐  →  정상 (residual 누적)
  da max가 65504 근접          →  float16 overflow 위험
""")

✓ 24개 블록 패치 완료
✓ Step tracker 부착 완료

1개 프롬프트 실행 중 (step 0만 캡처)...

검증 1: 재구성 오차  (L_minus_1 + delta_attn + delta_mlp == L_output?)
 Blk                 shape       err_sum     rel_sum    da/dm  판정
---------------------------------------------------------------------------
    0       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=1.416  ✓ 완벽
    1       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=0.334  ✓ 완벽
    2       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=1.514  ✓ 완벽
    3       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=0.761  ✓ 완벽
    4       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=1.186  ✓ 완벽
    5       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=2.214  ✓ 완벽
    6       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=0.807  ✓ 완벽
    7       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=1.181  ✓ 완벽
    8       (2, 4096, 1536)  err=0.000e+00  rel=0.00e+00  da/dm=1.291  ✓ 완벽
    9       (2, 4096, 1536)  err=0.000e+00 